In [ ]:
!nvidia-smi

Mon Feb 10 22:39:23 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   52C    P8             12W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
!nvcc --version

nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2024 NVIDIA Corporation
Built on Thu_Jun__6_02:18:23_PDT_2024
Cuda compilation tools, release 12.5, V12.5.82
Build cuda_12.5.r12.5/compiler.34385749_0


### Naive matrix transpose
Spin threads equal to total number of elements in the matrix

In [75]:
%%writefile matrixTranspose.cu
#include <stdio.h>

// Kernel function to perform inplace tranpose of a matrix
__global__ void matrixTranspose(float *A, int numRows, int numCols) {
    int row = blockIdx.y * blockDim.y + threadIdx.y;
    int col = blockIdx.x * blockDim.x + threadIdx.x;

    if (row < numRows && col < numCols) {
        int index = row * numCols + col;
        int transpose_index = col * numRows + row;
        int temp = A[index];
        A[index] = A[transpose_index];
        A[transpose_index] = temp;
    }
}


// Upper triangular transpose
__global__ void matrixOptimizedTranspose(float *A, int numRows, int numCols) {
    int row = blockIdx.y * blockDim.y + threadIdx.y;
    int col = blockIdx.x * blockDim.x + threadIdx.x;

    if (row < numRows && col < numCols && row<col) {
        int index = row * numCols + col;
        int transpose_index = col * numRows + row;
        int temp = A[index];
        A[index] = A[transpose_index];
        A[transpose_index] = temp;
        }
    }


void printMatrix(float *matrix, int numRows, int numCols) {
    for (int i = 0; i < numRows; ++i) {
        for (int j = 0; j < numCols; ++j) {
            printf("%.2f ", matrix[i * numCols + j]);
        }
        printf("\n");
    }
}

int main() {
    // Matrix dimensions
    int numRows = 128;  // Reduced for easier printing
    int numCols = 128;  // Reduced for easier printing
    int size = numRows * numCols * sizeof(float);

    // Host matrices
    float *h_A, *h_A_optimized;

    // Allocate host memory
    h_A = (float*)malloc(size);
    h_A_optimized = (float*)malloc(size);

    // Initialize host matrices
    for (int i = 0; i < numRows * numCols; ++i) {
      h_A[i] = (i)%numRows;
      h_A_optimized[i] = (i)%numRows;
    }
    // Print the matrices
    printf("Matrix A and A_optimized:\n");
    printMatrix(h_A, numRows, numCols);

    // Device matrices
    float *d_A;
    float *d_A_optimized;

    // Allocate device memory
    cudaMalloc((void**)&d_A, size);
    cudaMalloc((void**)&d_A_optimized, size);

    // Copy host matrices to device
    cudaMemcpy(d_A, h_A, size, cudaMemcpyHostToDevice);
    cudaMemcpy(d_A_optimized, h_A_optimized, size, cudaMemcpyHostToDevice);

    // Define grid and block dimensions
    dim3 threadsPerBlock(16, 16);
    dim3 numBlocks((numCols + threadsPerBlock.x - 1) / threadsPerBlock.x,
                   (numRows + threadsPerBlock.y - 1) / threadsPerBlock.y);

    // Launch the kernel
    matrixTranspose<<<numBlocks, threadsPerBlock>>>(d_A, numRows, numCols);
    cudaDeviceSynchronize();

    matrixOptimizedTranspose<<<numBlocks, threadsPerBlock>>>(d_A_optimized, numRows, numCols);
    cudaDeviceSynchronize();

    // Copy the result back to the host
    cudaMemcpy(h_A, d_A, size, cudaMemcpyDeviceToHost);
    cudaMemcpy(h_A_optimized, d_A_optimized, size, cudaMemcpyDeviceToHost);


    printf("\n Transposed in place: \n");
    printMatrix(h_A, numRows, numCols);

    printf("\n Transposed optimized: \n");
    printMatrix(h_A_optimized, numRows, numCols);

    // Free device memory
    cudaFree(d_A);
    cudaFree(d_A_optimized);

    // Free host memory
    free(h_A);
    free(h_A_optimized);

    return 0;
}

Overwriting matrixTranspose.cu


In [76]:
!nvcc -arch=compute_70 -code=sm_70 matrixTranspose.cu -o matTranspose

In [77]:
! ./matTranspose

Matrix A and A_optimized:
0.00 1.00 2.00 3.00 4.00 5.00 6.00 7.00 8.00 9.00 10.00 11.00 12.00 13.00 14.00 15.00 16.00 17.00 18.00 19.00 20.00 21.00 22.00 23.00 24.00 25.00 26.00 27.00 28.00 29.00 30.00 31.00 32.00 33.00 34.00 35.00 36.00 37.00 38.00 39.00 40.00 41.00 42.00 43.00 44.00 45.00 46.00 47.00 48.00 49.00 50.00 51.00 52.00 53.00 54.00 55.00 56.00 57.00 58.00 59.00 60.00 61.00 62.00 63.00 64.00 65.00 66.00 67.00 68.00 69.00 70.00 71.00 72.00 73.00 74.00 75.00 76.00 77.00 78.00 79.00 80.00 81.00 82.00 83.00 84.00 85.00 86.00 87.00 88.00 89.00 90.00 91.00 92.00 93.00 94.00 95.00 96.00 97.00 98.00 99.00 100.00 101.00 102.00 103.00 104.00 105.00 106.00 107.00 108.00 109.00 110.00 111.00 112.00 113.00 114.00 115.00 116.00 117.00 118.00 119.00 120.00 121.00 122.00 123.00 124.00 125.00 126.00 127.00 
0.00 1.00 2.00 3.00 4.00 5.00 6.00 7.00 8.00 9.00 10.00 11.00 12.00 13.00 14.00 15.00 16.00 17.00 18.00 19.00 20.00 21.00 22.00 23.00 24.00 25.00 26.00 27.00 28.00 29.00 30.00 31.00 32.00

In [78]:
!nvprof ./matTranspose

Matrix A and A_optimized:
0.00 1.00 2.00 3.00 4.00 5.00 6.00 7.00 8.00 9.00 10.00 11.00 12.00 13.00 14.00 15.00 16.00 17.00 18.00 19.00 20.00 21.00 22.00 23.00 24.00 25.00 26.00 27.00 28.00 29.00 30.00 31.00 32.00 33.00 34.00 35.00 36.00 37.00 38.00 39.00 40.00 41.00 42.00 43.00 44.00 45.00 46.00 47.00 48.00 49.00 50.00 51.00 52.00 53.00 54.00 55.00 56.00 57.00 58.00 59.00 60.00 61.00 62.00 63.00 64.00 65.00 66.00 67.00 68.00 69.00 70.00 71.00 72.00 73.00 74.00 75.00 76.00 77.00 78.00 79.00 80.00 81.00 82.00 83.00 84.00 85.00 86.00 87.00 88.00 89.00 90.00 91.00 92.00 93.00 94.00 95.00 96.00 97.00 98.00 99.00 100.00 101.00 102.00 103.00 104.00 105.00 106.00 107.00 108.00 109.00 110.00 111.00 112.00 113.00 114.00 115.00 116.00 117.00 118.00 119.00 120.00 121.00 122.00 123.00 124.00 125.00 126.00 127.00 
0.00 1.00 2.00 3.00 4.00 5.00 6.00 7.00 8.00 9.00 10.00 11.00 12.00 13.00 14.00 15.00 16.00 17.00 18.00 19.00 20.00 21.00 22.00 23.00 24.00 25.00 26.00 27.00 28.00 29.00 30.00 31.00 32.00